# Comparing log rotation approaches in Bash: inotifywait vs cron-driven polling

This notebook walks through two common strategies for rotating application logs with pure Bash tooling. The goal is to understand the trade-offs so you can pick the right approach for a given workload.

## What problem are we solving?

Long-running services write to stdout/stderr or to flat log files. Over time those files grow without bound, filling disk and making debugging slower. Rotation means:
- Detecting when a log file is ready to rotate.
- Renaming or compressing the old file.
- Signaling the application to reopen the new file (or relying on file-descriptor inheritance).
- Cleaning up old rotated files.

## Approach 1: cron-driven polling

A cron job runs a Bash script at fixed intervals (e.g., every 5 minutes). The script checks file size or age, rotates if needed, and prunes old archives. This is the simplest approach because it requires no persistent daemon and works on any system with cron.

In [ ]:
#!/usr/bin/env bash

# rotate-if-large.sh — cron-driven log rotation
# Intended to run from cron every 5 minutes.

set -o nounset
set -o pipefail

LOG_DIR="/var/log/myapp"
MAX_BYTES=$((10 * 1024 * 1024))  # 10 MB
MAX_AGE_DAYS=7
TIMESTAMP=$(date +%Y%m%d-%H%M%S)

mkdir -p "$LOG_DIR"

for f in "$LOG_DIR"/*.log; do
  [ -f "$f" ] || continue

  if [ $(stat -c%s "$f") -gt "$MAX_BYTES" ]; then
    mv "$f" "${f%.log}-${TIMESTAMP}.log"
    gzip "${f%.log}-${TIMESTAMP}.log"
    touch "$f"
    chown appuser:appgroup "$f"
  fi

  # Prune archives older than MAX_AGE_DAYS
  find "$LOG_DIR" -name "*.log.gz" -mtime +"$MAX_AGE_DAYS" -delete
done

echo "rotation check complete: $(date -Iseconds)"

### How to schedule it

Add a crontab entry:
```
*/5 * * * * /usr/local/bin/rotate-if-large.sh >> /var/log/myapp/rotation.log 2>&1
```

This approach is easy to reason about and easy to debug: just run the script by hand.

## Approach 2: inotifywait-driven rotation

`inotifywait` (from `inotify-tools`) blocks on an inotify watch and emits events when the kernel detects filesystem changes. A Bash loop can react to `CLOSE_WRITE` or `MODIFY` events, checking size after each write burst and rotating immediately rather than waiting for the next cron window.

In [ ]:
#!/usr/bin/env bash

# inotify-rotate.sh — event-driven log rotation
# Requires: inotify-tools (inotifywait)

set -o nounset
set -o pipefail

LOG_DIR="/var/log/myapp"
MAX_BYTES=$((10 * 1024 * 1024))
MAX_AGE_DAYS=7
WATCH_FILE="$LOG_DIR/app.log"

rotate_file() {
  local f="$1"
  [ -f "$f" ] || return 0
  local ts
  ts=$(date +%Y%m%d-%H%M%S)
  mv "$f" "${f%.log}-${ts}.log"
  gzip "${f%.log}-${ts}.log"
  touch "$f"
  chown appuser:appgroup "$f"
  echo "rotated $f -> ${f%.log}-${ts}.log"
}

prune_old() {
  find "$LOG_DIR" -name "*.log.gz" -mtime +"$MAX_AGE_DAYS" -delete
}

mkdir -p "$LOG_DIR"
touch "$WATCH_FILE"
chown appuser:appgroup "$WATCH_FILE"

# Loop forever; Ctrl-C to stop.
while true; do
  # Wait for a write-close event on the target file.
  inotifywait -e close_write --format "%f" "$LOG_DIR" 2>/dev/null | while IFS= read -r event_file; do
    [ "$event_file" = "app.log" ] || continue
    if [ $(stat -c%s "$WATCH_FILE") -gt "$MAX_BYTES" ]; then
      rotate_file "$WATCH_FILE"
      prune_old
    fi
  done
done

### How to run it

Start the watcher in the background or under a supervisor:
```
nohup /usr/local/bin/inotify-rotate.sh > /var/log/myapp/rotation.log 2>&1 &
```

This approach rotates closer to real time, but it requires `inotifywait` and a long-running Bash process.

## Comparison

| Dimension | cron-driven polling | inotifywait-driven |
|---|---|
| Latency | Up to the cron interval (e.g., 5 min) | Near-real-time on close_write |
| Dependencies | cron only | inotify-tools package |
| Failure mode | Missed rotation if cron is late | Watcher process can die silently |
| Disk usage | Log can grow between checks | Bounded by close_write cadence |
| Auditability | Easy: one-shot script logs to stdout | Harder: need to supervise the loop |

## Verify

1. **Cron path:** `bash rotate-if-large.sh && ls -lh /var/log/myapp/*.log.gz` — confirm an archive appeared.
2. **inotifywait path:** `inotifywait -m -e close_write /var/log/myapp` in one terminal, then `dd if=/dev/zero bs=1M count=11 of=/var/log/myapp/app.log` in another — confirm rotation fires within seconds.
3. **Pruning:** `find /var/log/myapp -name *.log.gz` — old files beyond `MAX_AGE_DAYS` should be absent.

## Common errors

- **cron PATH:** Cron runs with a minimal PATH. Always use absolute paths to `gzip`, `find`, and `date` inside cron scripts.
- **inotifywait on directories:** Watching the directory (`inotifywait -e close_write /var/log/myapp`) is more reliable than watching a single file, because `mv` breaks the watch on the old inode.
- **App SIGPIPE:** If the application holds the log file open across rotations, some programs continue writing to the unlinked inode until they reopen. Pair rotation with `systemctl reload app` or send `SIGUSR1` if the app supports reopen-on-signal.